In [1]:
import pandas as pd

In [17]:
movies = pd.read_csv("CLEANED_DATA\movies.csv")
genres = pd.read_csv("CLEANED_DATA\genres.csv")
cast = pd.read_csv("CLEANED_DATA\cast.csv")
crew = pd.read_csv("CLEANED_DATA\crew.csv")
movie_keywords = pd.read_csv("CLEANED_DATA\movie_keywords.csv")
movie_genres = pd.read_csv("CLEANED_DATA\movie_genres.csv")

# 10 Pandas Task Question

In [18]:
print("Unique Movies:", movies['movie_id'].nunique())

print("Unique Genres:", genres['genre_id'].nunique())

print("Unique Cast Members:", cast['person_id'].nunique())

print("Unique Crew Members:", crew['person_id'].nunique())

Unique Movies: 2503
Unique Genres: 19
Unique Cast Members: 11300
Unique Crew Members: 2848


In [19]:
missing_movies = movies[
    (movies['budget'].isnull()) |
    (movies['revenue'].isnull()) |
    (movies['budget'] == 0) |
    (movies['revenue'] == 0)
]

print("Total affected movies:", len(missing_movies))

missing_movies[
    ['movie_id', 'title', 'budget', 'revenue']
]

Total affected movies: 0


,movie_id,title,budget,revenue


In [20]:
valid_movies = movies[
    (movies['budget'] > 0) &
    (movies['revenue'] > 0)
]
valid_movies['profit'] = valid_movies['revenue'] - valid_movies['budget']
valid_movies['ROI'] = (valid_movies['profit'] / valid_movies['budget'])* 100
valid_movies[
    ['title', 'budget', 'revenue', 'profit', 'ROI']
].head(10)

,title,budget,revenue,profit,ROI
0,Interstellar,165000000.0,7.466067e+08,5.816067e+08,352.488913
1,Inception,160000000.0,8.390306e+08,6.790306e+08,424.394144
2,The Avengers,220000000.0,1.518816e+09,1.298816e+09,590.370689
3,The Dark Knight,185000000.0,1.004558e+09,8.195584e+08,443.004564
4,Avatar,237000000.0,2.923706e+09,2.686706e+09,1133.631235
5,Deadpool,58000000.0,7.828373e+08,7.248373e+08,1249.719564
6,Fight Club,63000000.0,1.008538e+08,3.785375e+07,60.085322
7,Avengers: Infinity War,300000000.0,2.052415e+09,1.752415e+09,584.138346
8,The Shawshank Redemption,25000000.0,2.834147e+07,3.341469e+06,13.365876
9,Pulp Fiction,8000000.0,2.139288e+08,2.059288e+08,2574.109525


In [21]:
high_budget_movies = valid_movies[
    valid_movies['budget'] >= 1000000
]
print("High-budget movies:", len(high_budget_movies))
top_15_movies = high_budget_movies.sort_values(by='ROI',ascending=False).head(15)
top_15_movies[['title', 'budget', 'revenue', 'profit', 'ROI']]

High-budget movies: 2457


,title,budget,revenue,profit,ROI
1068,Hush,1000000.0,136300000.0,135300000.0,13530.000000
1767,Dragon Ball Super: Broly,1000000.0,125002821.0,124002821.0,12400.282100
535,Snow White and the Seven Dwarfs,1488423.0,184925486.0,183437063.0,12324.256142
1791,The Rocky Horror Picture Show,1400000.0,171181400.0,169781400.0,12127.242857
467,Rocky,1000000.0,117253345.0,116253345.0,11625.334500
1227,Gone with the Wind,4000000.0,402352579.0,398352579.0,9958.814475
716,The Jungle Book,4000000.0,378000000.0,374000000.0,9350.000000
635,Cinderella,2900000.0,263600000.0,260700000.0,8989.655172
372,Saw,1200000.0,104045735.0,102845735.0,8570.477917
719,One Hundred and One Dalmatians,3600000.0,303000000.0,299400000.0,8316.666667


In [23]:
movies['release_date'] = pd.to_datetime(
    movies['release_date'],
    errors='coerce'
)

In [24]:
movies['release_year'] = movies['release_date'].dt.year
yearly_movies = movies['release_year'].value_counts().sort_index()
print(yearly_movies)
print("Highest movie output year:", yearly_movies.idxmax())
print("Number of movies:", yearly_movies.max())

release_year
1921     1
1922     1
1927     1
1931     2
1936     1
        ..
2022    65
2023    62
2024    43
2025    26
2026     8
Name: count, Length: 89, dtype: int64
Highest movie output year: 2016
Number of movies: 121


In [27]:
Q1 = movies['runtime'].quantile(0.25)
Q3 = movies['runtime'].quantile(0.75)
IQR = Q3 - Q1
print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = movies[
    (movies['runtime'] < lower) |
    (movies['runtime'] > upper)
]

print("Total outlier movies:", len(outliers))

outliers[['title', 'runtime']].sort_values(by='runtime',ascending=False)

Q1: 99.0
Q3: 126.0
IQR: 27.0
Total outlier movies: 60


,title,runtime
324,Zack Snyder's Justice League,242
1227,Gone with the Wind,238
829,Once Upon a Time in America,229
1610,Lawrence of Arabia,228
1774,Ben-Hur,212
587,The Irishman,209
1277,Seven Samurai,207
1304,Killers of the Flower Moon,206
187,The Godfather Part II,202
20,The Lord of the Rings: The Return of the King,201


In [29]:
movie_genres.head(10)

,movie_id,genre_id
0,157336,12
1,157336,18
2,157336,878
3,27205,28
4,27205,878
5,27205,12
6,24428,878
7,24428,28
8,24428,12
9,155,28


In [32]:
# Get the first genre for each movie
primary_genres = movie_genres.drop_duplicates(
    subset='movie_id',
    keep='first'
)

# Get genre names
primary_genres = primary_genres.merge(
    genres,
    on='genre_id',
    how='left'
)

# Get movie title and popularity
primary_genres = primary_genres.merge(
    movies[['movie_id', 'title', 'popularity']],
    on='movie_id',
    how='left'
)

# Calculate average popularity for each genre
genre_popularity = primary_genres.groupby(
    'genre_name'
)['popularity'].mean().sort_values(ascending=False)

print(genre_popularity)

genre_name
Science Fiction    21.261836
Music              18.324340
Animation          16.811212
Family             16.710635
Action             16.385188
War                15.470317
Horror             14.851835
Adventure          14.785510
Fantasy            13.022920
Mystery            12.427131
Western            12.322680
Thriller           11.294211
Drama              11.278534
Crime              10.739498
History            10.414567
Comedy              9.984262
Romance             9.976708
Documentary         7.726200
Name: popularity, dtype: float64


In [33]:
actor_movies = cast.merge(
    movies[['movie_id', 'vote_average']],
    on='movie_id',
    how='inner'
)
actor_stats = actor_movies.groupby('actor_name').agg(
    movie_count=('movie_id', 'nunique'),
    average_rating=('vote_average', 'mean')
)
top_10_actors = actor_stats.sort_values(
    by='movie_count',
    ascending=False
).head(10)

print(top_10_actors)

                    movie_count  average_rating
actor_name                                     
Samuel L. Jackson            39        6.938692
Robert De Niro               38        7.228316
Brad Pitt                    38        7.257289
Johnny Depp                  37        6.856973
Tom Hanks                    33        7.289758
Willem Dafoe                 32        7.114781
Mark Wahlberg                32        6.630063
Scarlett Johansson           32        7.056344
Tom Cruise                   31        7.040613
Morgan Freeman               31        6.963742


In [34]:
# Find movies with duplicate titles
duplicate_movies = movies[
    movies['title'].duplicated(keep=False)
].copy()

# Extract release year
duplicate_movies['release_year'] = (
    duplicate_movies['release_date'].dt.year
)

# Sort and display results
duplicate_movies = duplicate_movies.sort_values(
    by=['title', 'release_year']
)

print("Total duplicate title records:", len(duplicate_movies))

print(duplicate_movies[
    ['title', 'release_year']
])

Total duplicate title records: 100
                          title  release_year
878   A Nightmare on Elm Street          1984
1928  A Nightmare on Elm Street          2010
261                     Aladdin          1992
339                     Aladdin          2019
792         Alice in Wonderland          1951
...                         ...           ...
1737                  The Thing          2011
744                Total Recall          1990
860                Total Recall          2012
2500               Transformers          1987
252                Transformers          2007

[100 rows x 2 columns]


In [36]:
correlation = movies['popularity'].corr(movies['vote_count'])
print("Correlation:", correlation)
pop_median = movies['popularity'].median()
vote_median = movies['vote_count'].median()
print("Popularity median:", pop_median)
print("Vote count median:", vote_median)
unusual_movies = movies[
    (
        (movies['popularity'] > pop_median) &
        (movies['vote_count'] < vote_median)
    ) |
    (
        (movies['popularity'] < pop_median) &
        (movies['vote_count'] > vote_median)
    )
]

print("Unusual movies:", len(unusual_movies))

print(unusual_movies[
    ['title', 'popularity', 'vote_count']
].sort_values(by='popularity', ascending=False))

Correlation: 0.36085002849007364
Popularity median: 10.3971
Vote count median: 4361.0
Unusual movies: 641
                              title  popularity  vote_count
2388                 Disclosure Day    394.3346        2328
1315                      Obsession    364.8722        4203
2248                      Backrooms    258.4743        2483
1465                        Michael    108.7248        3846
2452               Mortal Kombat II     99.5231        2260
...                             ...         ...         ...
322                              X2      0.8734       11013
417   The Lost World: Jurassic Park      0.8620        9356
547               Jurassic Park III      0.7820        7968
1197                 The Internship      0.6335        4527
731          Grave of the Fireflies      0.0170        6646

[641 rows x 3 columns]
